# Домашняя работа 2 — качество данных

## Подготовка

In [1]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
LOCAL = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
print("pandas", pd.__version__)

pandas 2.2.3


## Независимая загрузка

In [2]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
export = pd.read_csv(LOCAL / "training_export.csv", dtype="string")
raw_copy = export.copy(deep=True)
print(decisions.shape, export.shape)
print(export.dtypes)
export.head(5)

(6830, 7) (6832, 7)
source_row       string[python]
activity_year    string[python]
lei              string[python]
county_code      string[python]
action_taken     string[python]
loan_amount      string[python]
income           string[python]
dtype: object


,source_row,activity_year,lei,county_code,action_taken,loan_amount,income
0,2,2023,549300VZVN841I2ILS84,11001,1,oops,72.0
1,3,2023,549300P1347BFCBHJV60,11001,88,585000.0,174.0
2,5,2023,549300P1347BFCBHJV60,11001,1,275000.0,BAD
3,7,2023,549300RPOGWJRH63HS39,11001,1,295000.0,63.0
4,8,2023,549300RPOGWJRH63HS39,11001,1,1125000.0,247.0


## H1. Собственная функция очистки

In [3]:
def clean_export(source):
    df = source.copy(deep=True)

    # Проверка и удаление повторных записей
    has_key = df["source_row"].notna()
    unique_records = df.loc[has_key].drop_duplicates()

    assert unique_records["source_row"].is_unique, "Конфликтующие версии ключа"

    duplicates = has_key & df["source_row"].duplicated()
    replay_count = int(duplicates.sum())
    df = df.loc[~duplicates].copy()

    # Очистка кода округа
    df["county_code"] = df["county_code"].str.strip()

    # Преобразование числовых столбцов
    parse_errors = {}
    numeric_columns = ["source_row", "loan_amount", "income", "action_taken"]

    for column in numeric_columns:
        converted = pd.to_numeric(df[column], errors="coerce")
        parse_errors[column] = int(
            (df[column].notna() & converted.isna()).sum()
        )
        df[column] = converted

    # Проверка корректности записей
    reasons = pd.Series("", index=df.index, dtype="string")

    checks = {
        "unknown_source_row": df["source_row"].isna(),
        "invalid_loan_amount": df["loan_amount"].isna() | (df["loan_amount"] <= 0),
        "invalid_action_taken": ~df["action_taken"].isin([1, 2, 3]),
        "invalid_county": ~df["county_code"].eq("11001").fillna(False),
    }

    for name, mask in checks.items():
        reasons.loc[mask] += name + ";"

    # Разделение корректных и ошибочных записей
    invalid_mask = reasons.ne("")

    clean = df.loc[~invalid_mask].copy()
    quarantine = source.loc[df.index[invalid_mask]].copy()
    quarantine["reason"] = reasons.loc[invalid_mask]

    # Формирование отчёта
    audit = {
        "input": len(source),
        "replays": replay_count,
        "clean": len(clean),
        "quarantine": len(quarantine),
        "parse_errors": parse_errors,
    }

    assert len(source) == replay_count + len(clean) + len(quarantine)

    return clean, quarantine, audit

## H2. Год и известный доход

In [4]:
clean, quarantine, audit = clean_export(export)

assert clean["activity_year"].eq("2023").fillna(False).all()

income = clean["income"]

print("Количество очищенных записей:", len(clean))
print("Записи с известным доходом:", income.notna().sum())
print("Записи без дохода:", income.isna().sum())
print("Средний доход, тыс. долларов в год:", income.mean())
print("Результаты очистки:", audit)

total = audit["replays"] + audit["clean"] + audit["quarantine"]

assert audit["input"] == total
assert_frame_equal(export, raw_copy)

Количество очищенных записей: 6828
Записи с известным доходом: 6485
Записи без дохода: 343
Средний доход, тыс. долларов в год: 237.06553585196608
Результаты очистки: {'input': 6832, 'replays': 2, 'clean': 6828, 'quarantine': 2, 'parse_errors': {'source_row': 0, 'loan_amount': 1, 'income': 1, 'action_taken': 0}}


## H3. Проверка отчётного года

In [5]:
test_export = export.copy(deep=True)

first_index = test_export.index[0]
test_export.loc[first_index, "activity_year"] = "2022"

valid_year = test_export["activity_year"].eq("2023").fillna(False).all()

print("Проверка года:", valid_year)

assert valid_year == False
assert_frame_equal(export, raw_copy)

Проверка года: False


## H+1. Конфликт исходных значений

## H+2. Совпадение опубликованных полей